# 05. LLaMA3 Block Tutorial | LLaMA3 Block 教程

**难度：** Medium | **环境：** [CPU 主线](#runtime-environment) | **标签：** `模型结构`, `Transformer`, `LLaMA Block` | **目标人群：** 熟悉注意力与前馈网络的学习者

---

## 本节导读

前面的 RMSNorm、SwiGLU、RoPE 和 GQA 分别处理尺度、非线性、位置和上下文；进入模型结构后，它们需要在同一个解码器层中协同工作。理解这个组合顺序，才能读懂现代仅解码器模型如何逐层更新主状态。

本节沿着 LLaMA 类解码器层跟踪组件的组合顺序：两次前置归一化（Pre-Norm）、两条变换分支和两次残差更新。学习后应能沿着“主状态 → 注意力更新 → MLP 更新”追踪每一步的输入、输出与形状，并判断改动某个组件会影响哪条路径。

**关键词：** `LLaMA3`, `Transformer Block`, `Decoder Layer`

---

<a id="runtime-environment"></a>

## 运行环境

基础练习使用 [BASE-CPU](../docs/guide.md#base-cpu)，不需要 GPU。

## 前置阅读

进入本节前，先能指出 RMSNorm、RoPE、GQA 和 SwiGLU 在解码器层中分别承担什么作用，再把它们按执行顺序组合起来。

- [01. RMSNorm Tutorial | RMSNorm 教程](../02_PyTorch_Algorithms/01_RMSNorm_Tutorial.ipynb)
- [02. SwiGLU Activation | SwiGLU 激活](../02_PyTorch_Algorithms/02_SwiGLU_Activation.ipynb)
- [04. Attention MHA GQA | 多头注意力](../02_PyTorch_Algorithms/04_Attention_MHA_GQA.ipynb)


---

## Step 1：解码器块的组成、状态流与层堆叠

主状态是每个词元当前的隐藏向量；残差流是它在各层间持续传递并接收增量的路径。仅解码器模型先把 token（词元）转成主状态，再让多个解码器块依次更新同一条残差流。单个块包含两类变换：注意力在 token 之间交换信息，前馈网络（MLP）在每个 token 内重组通道；归一化控制子层输入尺度，残差连接保留主状态并接收增量更新。

| 组件 | 读取什么 | 写回什么 | 跨多个块的意义 |
|---|---|---|---|
| 前置归一化 | 当前主状态 | 尺度受控的子层输入 | 防止每层直接接收失控尺度 |
| 注意力 | 当前 token 与可见上下文 | 上下文更新量 | 逐层扩大和重组 token 间信息 |
| MLP / SwiGLU | 单个 token 的通道 | 通道更新量 | 增加逐 token 的非线性容量 |
| 残差主干 | 块输入与两个更新量 | 下一块的主状态 | 为深层网络保留连续的信息与梯度路径 |

![LLaMA3 Decoder Block 与多层残差主干](../docs/public/02_PyTorch_Algorithms/05_llama_block.svg)


## Step 2：两次前置归一化残差更新

设块输入 $h$ 的形状为 $[B,S,C]$，其中 $B$ 是批量大小、 $S$ 是词元数、 $C$ 是隐藏宽度； $h'$ 是第一次残差更新后的状态， $h_{next}$ 是块输出。一个块依次执行两次“归一化 → 子层变换 → 残差相加”。注意力和 MLP 解决的问题不同，因此不能把两条更新合并成一次无序求和；第二次前置归一化必须读取已经包含注意力更新的状态。

$$
h' = h + \mathrm{Attention}(\mathrm{Norm}_1(h)),\qquad
h_{next}=h' + \mathrm{MLP}(\mathrm{Norm}_2(h')).
$$

| 路径 | 输入状态 | 更新来源 | 需要保持的不变量 |
|---|---|---|---|
| 注意力更新 | `h` | token 间上下文 | 输出形状与 `h` 一致 |
| MLP 更新 | `h'` | token 内通道变换 | 必须读取第一次残差后的状态 |
| 块输出 | 两次残差后的状态 | 主干与增量共同构成 | 可直接作为下一块输入 |

![两次 Pre-Norm 残差更新](../docs/public/02_PyTorch_Algorithms/05_prenorm_residual_updates.svg)


## Step 3：组件替换如何保持输入输出与残差接口

多头注意力（MHA）、分组查询注意力（GQA）和潜在注意力（MLA）采用不同的头组织或历史状态表示；混合专家（MoE）则按词元选择前馈分支。架构演进经常替换注意力、归一化或 MLP，但块外部仍要维持主状态的形状、数据类型和残差接口。局部组件能够产生同形状输出，只说明接口可连接；深度增加还会放大初始化、残差尺度、归一化位置和梯度传播之间的相互作用。

| 结构改动 | 块内必须保持 | 堆叠后需要观察 |
|---|---|---|
| MHA → GQA / MLA | 注意力输出仍回到隐藏宽度 | 状态容量、质量和执行后端路径 |
| 稠密 MLP → MoE | token 输出仍回到原位置与通道 | 路由集中、容量溢出与通信 |
| LayerNorm → RMSNorm | 归一化输入输出形状不变 | 激活尺度、梯度尺度与收敛 |
| 残差缩放或初始化变化 | 两次更新顺序不变 | 深层激活增长、梯度衰减或爆炸 |


## Step 4：代码设计与解码器块组装

本题用 `StructureNormStub` 的逐通道缩放和 `AttentionUpdateStub` 的线性映射作为结构替身，分别提供同形状输出；它们不是 RMSNorm 或真实注意力。SwiGLU 保留门控前馈计算。题目关注解码器块的组合责任：依次写回两次残差更新，并让多个块顺序更新同一条主状态。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|---|---|---|---|
| TODO 1 · `LlamaDecoderLayer.forward` | 写回注意力残差更新 | 先前置归一化，再把上下文增量加回主状态 | MLP 接收第一次更新后的状态 |
| TODO 2 · `LlamaDecoderLayer.forward` | 写回 MLP 残差更新 | 必须读取第一次更新后的状态 | 输出形状、调用顺序与梯度连通性正确 |
| TODO 3 · `DecoderStack.forward` | 让多个块顺序更新主状态 | 后一层读取前一层输出，不得并行覆盖 | 固定增量按层累积，最终形状不变 |

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
class StructureNormStub(nn.Module):
    """记录 Pre-Norm 调用位置的结构替身。

    它只保留形状与可训练缩放接口，不实现 RMSNorm 的均方根统计；
    RMSNorm 的数值机制由第 01 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return x * self.scale


class AttentionUpdateStub(nn.Module):
    """产生同形状的上下文更新，用于验证 Block 组装。

    不实现 causal mask、RoPE、MHA 或 GQA；这些机制分别由第 03、04 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return self.proj(x)


class LlamaMLP(nn.Module):
    """作为固定依赖提供最小 SwiGLU 通道更新。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)


class LlamaDecoderLayer(nn.Module):
    """组装最小 Pre-Norm Decoder Block，验证两次残差更新顺序。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.norm_1 = StructureNormStub(hidden_size)
        self.attention_update = AttentionUpdateStub(hidden_size)
        self.norm_2 = StructureNormStub(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """返回与输入同形状的 Decoder Block 输出。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
        """
        # TODO 1：先计算 Attention 更新，并将其写回原始 hidden_states。
        attention_delta = self.attention_update(self.norm_1(hidden_states))
        hidden_states = hidden_states + attention_delta


        # TODO 2：基于第一次更新后的 hidden_states 计算 MLP 更新并写回。
        mlp_delta = self.mlp(self.norm_2(hidden_states))
        hidden_states = hidden_states + mlp_delta
        return hidden_states


class DecoderStack(nn.Module):
    """让多个 Decoder Block 顺序更新同一条隐藏状态。"""

    def __init__(self, hidden_size: int, intermediate_size: int, num_layers: int):
        super().__init__()
        if num_layers <= 0:
            raise ValueError("num_layers 必须为正整数")
        self.layers = nn.ModuleList(
            [LlamaDecoderLayer(hidden_size, intermediate_size) for _ in range(num_layers)]
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """按层顺序返回最终隐藏状态。"""
        # TODO 3：让每层读取上一层的输出。
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        return hidden_states


## Step 5：前沿模型如何改写同一个解码器块

前面的 Step 1–4 描述的是 LLaMA3 形态的解码器块：一条主状态、两次前置归一化、两次残差相加。2026 年的前沿模型并没有推翻这个骨架，而是在同一条骨架上替换内部组件。理解这些改动的价值在于看清「哪些接口必须保持」和「哪些机制被替换」这两件事的分离。

块对外仍然只接受 `[B, S, C]` 的主状态、返回同形状的主状态；被替换的是块内部的注意力读取范围、MLP 的参数组织方式，以及残差主干从单流变多流之后的混合方式。

本节用两个实例对照这条边界：

- **Qwen3** 保留完整骨架，只在注意力内部做局部替换（逐头归一化、滑动窗口交替）。
- **DeepSeek-V4** 同时重写注意力、MLP 与残差接口三处组件。

| 模型 | 改动落点 | 主干形态 | 残差形式 | 块外接口 |
|---|---|---|---|---|
| LLaMA3 | 基准 | 单流 `[B, S, D]` | `h + delta` | 不变 |
| Qwen3 | 注意力内部 | 单流 `[B, S, D]` | `h + delta` | 不变 |
| DeepSeek-V4 | 注意力、MLP、残差 | N 流 `[B, S, N, D]` | `post ⊗ out + Comb^T @ h` | 不变 |

### 5.1 Qwen3：保留骨架的局部替换

Qwen3 的解码器块与 LLaMA3 同构：两次前置归一化、两次残差相加、注意力在前、MLP 在后。它的创新全部落在注意力子层内部，且不改变该子层的输入输出形状。

| Qwen3 组件 | 替换了什么 | 块内必须保持 | 需要额外验证的机制 |
|---|---|---|---|
| 逐头归一化（QK-Norm） | 查询与键在进入点积前的尺度 | 归一化只在 `head_dim` 上进行 | 归一化后注意力分数的尺度是否回到可训练区间 |
| 滑动窗口交替层 | 注意力的历史访问范围 | 注意力输出仍回到隐藏宽度 | 窗口起点随查询位置前移，因果性仍然成立 |
| 分组查询注意力（GQA） | 键值头的数量 | 键值扩展后头数与查询头数一致 | 缓存保存紧凑的键值而非扩展后的结果 |

逐头归一化解决的是一个具体的训练故障：模型规模变大后查询与键的内积尺度随之增长，注意力分数变得过大，经过 softmax 后接近独热分布，梯度随之消失。归一化把分数尺度拉回可训练区间，而它只需作用在 `head_dim` 这一维，不改变任何张量形状。

滑动窗口交替层解决的是另一个故障：键值缓存随序列长度线性增长，长上下文下显存占用不可接受。让一半的层只访问最近的窗口，缓存规模随之减半；另一半的层保留完整访问范围负责全局依赖，两者交替出现。

### 5.2 DeepSeek-V4：骨架本身被重写

V4 的改动落点更多，同样不改变块外接口。

| V4 组件 | 替换了什么 | 块外必须保持 | 需要额外验证的机制 |
|---|---|---|---|
| 压缩稀疏注意力（CSA / HCA） | 注意力的历史表示与访问范围 | 注意力输出仍回到隐藏宽度 | 压缩窗口的因果边界、压缩条目的位置编码 |
| Lightning Indexer | 压缩条目之间的选择方式 | 选中数量与形状不变 | top-k 不能选中因果上尚未就绪的条目 |
| 注意力汇（Attention Sink） | softmax 的归一化范围 | 输出形状与普通注意力相同 | 汇参与真实键共享同一份概率质量 |
| 分组低秩输出投影 | 注意力输出到隐藏宽度的映射方式 | 输出仍为 `[B, S, C]` | 分组数量与低秩维度的乘积关系 |
| 稀疏混合专家（MoE） | MLP 的参数组织与词元分发 | 词元输出回到原位与原通道 | 路由集中度、共享专家的恒定参与 |
| 多流超连接（mHC） | 残差主干的形态与混合方式 | 块输入输出形状不变 | 混合矩阵的双随机性与无穷范数非扩张 |

两类改动的共同约束是块外接口不变。Qwen3 证明局部替换足以拿到可观的收益；V4 证明残差主干本身也可以被重写，但需要额外承担主路约束的验证成本。


## 测试

用记录输入的结构替身核对两次残差的读取顺序，再检查梯度连通性和多层累积。

In [3]:
# 测试设计：分别验证两次残差顺序、梯度连通性与多层堆叠。


class ConstantUpdate(nn.Module):
    """为残差顺序测试提供可辨识、同形状的固定更新。"""

    def __init__(self, value: float):
        super().__init__()
        self.value = value
        self.seen_input = None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.seen_input = x.detach().clone()
        return torch.full_like(x, self.value)


def test_residual_update_order():
    """验证 MLP 读取 Attention 更新后的状态，并按顺序累加两次更新。"""
    layer = LlamaDecoderLayer(hidden_size=4, intermediate_size=6)
    layer.norm_1 = nn.Identity()
    layer.attention_update = ConstantUpdate(2.0)
    layer.norm_2 = nn.Identity()
    layer.mlp = ConstantUpdate(3.0)
    x = torch.ones(1, 2, 4)

    out = layer(x)
    first_update = x + 2.0
    assert torch.allclose(layer.mlp.seen_input, first_update), "MLP 必须接收第一次残差更新后的状态"
    assert torch.allclose(out, x + 5.0), "两次残差更新必须按 Attention → MLP 的顺序累加"


def test_block_shape_calls_and_gradients():
    """验证单个 Block 的形状、组件调用次数和参数梯度。"""
    torch.manual_seed(1)
    layer = LlamaDecoderLayer(hidden_size=8, intermediate_size=12)
    x = torch.randn(2, 3, 8, requires_grad=True)
    out = layer(x)
    assert out.shape == x.shape
    assert layer.norm_1.calls == 1 and layer.attention_update.calls == 1 and layer.norm_2.calls == 1
    out.sum().backward()
    assert all(param.grad is not None for param in layer.parameters())



class AdditiveBlock(nn.Module):
    """为层堆叠测试提供可辨识的固定增量。"""

    def __init__(self, value: float):
        super().__init__()
        self.value = value

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        return hidden_states + self.value


def test_decoder_stack_order():
    """验证多个 Block 的输出按层传递，而不是并行覆盖。"""
    stack = DecoderStack(hidden_size=4, intermediate_size=6, num_layers=3)
    stack.layers = nn.ModuleList([AdditiveBlock(1.0), AdditiveBlock(2.0), AdditiveBlock(3.0)])
    x = torch.zeros(1, 2, 4)
    assert torch.allclose(stack(x), torch.full_like(x, 6.0))


def run_tests():
    """运行 Decoder Block 组装与层堆叠测试。"""
    test_residual_update_order()
    test_block_shape_calls_and_gradients()
    test_decoder_stack_order()
    print("✅ Decoder Block：残差顺序、梯度连通与层堆叠测试通过")


try:
    run_tests()
except NotImplementedError:
    print("请先按 TODO 1–3 完成 Block 组装，再运行测试。")
    raise


✅ Decoder Block：残差顺序、梯度连通与层堆叠测试通过


🛑 **STOP HERE** 🛑

请先完成题目区并通过测试，再继续查看参考代码与解析。

## 参考代码与解析

### 代码

In [4]:
class StructureNormStub(nn.Module):
    """记录 Pre-Norm 调用位置的结构替身。

    它只保留形状与可训练缩放接口，不实现 RMSNorm 的均方根统计；
    RMSNorm 的数值机制由第 01 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(dim))
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return x * self.scale


class AttentionUpdateStub(nn.Module):
    """产生同形状的上下文更新，用于验证 Block 组装。

    不实现 causal mask、RoPE、MHA 或 GQA；这些机制分别由第 03、04 节验证。
    """

    def __init__(self, dim: int):
        super().__init__()
        self.proj = nn.Linear(dim, dim, bias=False)
        self.calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.calls += 1
        return self.proj(x)


class LlamaMLP(nn.Module):
    """保留真实 SwiGLU 门控计算的最小 MLP。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        # 固定依赖：SwiGLU 的三个无 bias 投影。
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # 固定依赖：先计算门控通道更新，再回到 hidden_size。
        gate = F.silu(self.gate_proj(x))
        up = self.up_proj(x)
        return self.down_proj(gate * up)


class LlamaDecoderLayer(nn.Module):
    """组装最小 Pre-Norm Decoder Block，验证两次残差更新顺序。"""

    def __init__(self, hidden_size: int, intermediate_size: int):
        super().__init__()
        self.norm_1 = StructureNormStub(hidden_size)
        self.attention_update = AttentionUpdateStub(hidden_size)
        self.norm_2 = StructureNormStub(hidden_size)
        self.mlp = LlamaMLP(hidden_size, intermediate_size)

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """返回与输入同形状的 Decoder Block 输出。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
        """
        # TODO 1：先计算 Attention 更新，并将其写回原始 hidden_states。
        attention_delta = self.attention_update(self.norm_1(hidden_states))
        hidden_states = hidden_states + attention_delta

        # TODO 2：基于第一次更新后的 hidden_states 计算 MLP 更新并写回。
        mlp_delta = self.mlp(self.norm_2(hidden_states))
        hidden_states = hidden_states + mlp_delta
        return hidden_states


class DecoderStack(nn.Module):
    """让多个 Decoder Block 顺序更新同一条隐藏状态。"""

    def __init__(self, hidden_size: int, intermediate_size: int, num_layers: int):
        super().__init__()
        if num_layers <= 0:
            raise ValueError("num_layers 必须为正整数")
        self.layers = nn.ModuleList(
            [LlamaDecoderLayer(hidden_size, intermediate_size) for _ in range(num_layers)]
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """按层顺序返回最终隐藏状态。"""
        # TODO 3：让每层读取上一层的输出。
        for layer in self.layers:
            hidden_states = layer(hidden_states)
        return hidden_states


### 解析

#### TODO 1：写回注意力残差更新

- 上下文分支读取归一化后的输入，但更新必须加回原始主状态。
- 生成同形状注意力增量并完成第一次相加；顺序测试记录 MLP 实际看到的输入。

#### TODO 2：写回 MLP 残差更新

- 第二次前置归一化必须读取已包含注意力更新的中间状态。
- 生成通道增量并完成第二次相加；测试核对输出形状、调用次数和梯度连通性。

#### TODO 3：顺序堆叠多个块

- 第 $l+1$ 层的输入必须是第 $l$ 层的完整输出。
- 逐层覆盖当前隐藏状态；固定增量测试检查所有层按顺序累积。

## 可选 CPU 延伸：把一个块接到最小语言模型

词元编号先经嵌入层（Embedding）变成隐藏向量，经过解码器块后，语言模型输出层（LM Head）为词表中每个词元生成分数（logits），再与目标词元计算损失（loss）。下面用随机权重在 CPU 上检查这条数据流和梯度是否连通。

In [5]:
from typing import Optional
class TinyLlamaLM(nn.Module):
    """
        最小 LLaMA 语言模型，仅用于验证 DecoderLayer 结构正确性。

        结构：Embedding → 1 × DecoderLayer → LM Head
        注意：此实现仅为教学验证用途，非生产级模型。
    """
    def __init__(self, vocab_size: int, hidden_size: int, intermediate_size: int):
        super().__init__()
        # 最小 LM 只保留三层：Embedding -> Decoder Layer -> LM Head
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.decoder = LlamaDecoderLayer(hidden_size, intermediate_size)
        self.lm_head = nn.Linear(hidden_size, vocab_size, bias=False)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        # 1) 把 token id 映射成 hidden states
        hidden_states = self.embed_tokens(input_ids)
        # 2) 经过前面已经组装好的 LLaMA-3 Decoder Layer
        hidden_states = self.decoder(hidden_states)
        # 3) 投影到词表维度，得到 next-token logits
        logits = self.lm_head(hidden_states)
        return logits


def build_toy_batch(
    batch_size: int = 2,
    seq_len: int = 4,
    vocab_size: int = 16,
    device: torch.device = None,
    ):
    """
    构造一个 toy batch，用于最小训练闭环验证。

    Args:
        batch_size: 批次大小
        seq_len: 序列长度
        vocab_size: 词表大小，生成的 token id 在 [0, vocab_size-1] 范围内
        device: 张量所在设备

    Returns:
        input_ids: [batch_size, seq_len]
        labels: [batch_size, seq_len]
    """
    # 一个极小的 next-token toy batch，用来演示最小训练闭环。
    # input_ids 是模型输入
    input_ids = torch.randint(0, vocab_size, (batch_size, seq_len), device=device)
    # labels 是 input_ids 的简单右移版本
    labels = torch.roll(input_ids, shifts=-1, dims=1)
    # 将最后一个位置的标签设为 0（避免越界，实际训练中会 mask 掉）
    labels[:, -1] = 0
    return input_ids, labels


def train_one_step(
    model: nn.Module, 
    batch: tuple[torch.Tensor, torch.Tensor],
    optimizer: torch.optim.Optimizer,
    )-> tuple[torch.Tensor, torch.Tensor]:
    """
    执行一步训练。

    Args:
        model: 要训练的模型
        batch: (input_ids, labels)
        optimizer: 优化器

    Returns:
        loss: 当前步的 loss
        logits: 模型输出 logits
    """
    input_ids, labels = batch
    # 前向：得到每个位置上的 next-token logits
    logits = model(input_ids)

    # 预测 t+1 token（因果语言建模）
    shift_logits = logits[:, :-1].reshape(-1, logits.size(-1))
    shift_labels = labels[:, 1:].reshape(-1)

    # 只保留 next-token 监督关系：预测 t+1 的 token
    loss = F.cross_entropy(shift_logits, shift_labels)

    # 标准训练三连：清梯度 -> 反传 -> 更新参数
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss, logits


def run_minimal_validation(
    vocab_size: int = 16,
    hidden_size: int = 32,
    intermediate_size: int = 80,
    batch_size: int = 2,
    seq_len: int = 4,
    num_steps: int = 10,
    lr: float = 1e-3,
    device: Optional[torch.device] = None,
) -> None:
    """
    运行最小训练闭环验证。

    Args:
        vocab_size: 词表大小
        hidden_size: 隐藏层维度
        intermediate_size: SwiGLU 中间层维度
        batch_size: 批次大小
        seq_len: 序列长度
        num_steps: 训练步数
        lr: 学习率
        device: 张量所在设备
    """
    print("=== 最小训练闭环验证 ===")
    print(f"配置: vocab_size={vocab_size}, hidden_size={hidden_size}, "
          f"intermediate_size={intermediate_size}, batch_size={batch_size}, "
          f"seq_len={seq_len}, num_steps={num_steps}")

    torch.manual_seed(42)

    model = TinyLlamaLM(vocab_size, hidden_size, intermediate_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    batch = build_toy_batch(batch_size, seq_len, vocab_size)

    losses = []

    for step in range(num_steps):
        loss, logits = train_one_step(model, batch, optimizer)
        losses.append(float(loss))
        if step % 2 == 0 or step == num_steps - 1:
            print(f"  Step {step+1:2d}: loss = {loss.item():.6f}")

    # 验证
    assert logits.shape == (batch_size, seq_len, vocab_size), \
        f"Expected logits shape {(batch_size, seq_len, vocab_size)}, got {logits.shape}"
    assert torch.isfinite(loss), f"Loss is not finite: {loss.item()}"

    # 验证 loss 在下降
    if losses[0] > losses[-1]:
        print(f"✅ Loss 下降: {losses[0]:.6f} → {losses[-1]:.6f}")
    else:
        print(f"⚠️ Loss 未明显下降: {losses[0]:.6f} → {losses[-1]:.6f}")

    print("✅ 最小数据逻辑与训练循环通过\n")

if __name__ == "__main__":
    # 运行验证
    run_minimal_validation()

=== 最小训练闭环验证 ===
配置: vocab_size=16, hidden_size=32, intermediate_size=80, batch_size=2, seq_len=4, num_steps=10


  Step  1: loss = 3.182898
  Step  3: loss = 3.006450
  Step  5: loss = 2.837282
  Step  7: loss = 2.672900
  Step  9: loss = 2.511093
  Step 10: loss = 2.430521
✅ Loss 下降: 3.182898 → 2.430521
✅ 最小数据逻辑与训练循环通过



/var/folders/34/q_h8w8w92b160jvbcnw4598c0000gn/T/ipykernel_59695/3796061543.py:128: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:821.)
  losses.append(float(loss))


In [6]:
class HeadDimRMSNorm(nn.Module):
    """沿 `head_dim` 归一化，用于校准查询与键在点积前的尺度。

    与常规 RMSNorm 的差别只在于作用维度：常规用法归一化整个隐藏宽度，
    而这里归一化每个注意力头自己的 `head_dim`。作用在 `head_dim` 上意味着
    每个头独立校准，头的语义不会被其他头混合干扰。
    """

    def __init__(self, head_dim: int, eps: float = 1e-6):
        super().__init__()
        if head_dim <= 0:
            raise ValueError('head_dim 必须为正数')
        self.head_dim = head_dim
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(head_dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """对最后一维做 RMS 归一化并施加逐通道缩放。

        Args:
            x: 形状 `[..., head_dim]` 的张量。
        Returns:
            与输入同形状的归一化结果。
        """
        x_fp32 = x.float()
        mean_square = x_fp32.pow(2).mean(dim=-1, keepdim=True)
        normalized = x_fp32 * torch.rsqrt(mean_square + self.eps)
        return (self.weight * normalized).to(dtype=x.dtype)


class QKNormAttention(nn.Module):
    """带逐头归一化的分组查询注意力。

    查询与键在投影并拆成多头之后、进入点积之前各自做一次沿 `head_dim` 的
    归一化。该归一化不改变任何张量形状，但会把注意力分数的尺度拉回可训练
    区间，避免大模型训练中 softmax 退化为独热分布而丢失梯度。
    """

    def __init__(self, hidden_size: int, num_heads: int, num_kv_heads: int = None):
        super().__init__()
        if hidden_size <= 0 or num_heads <= 0:
            raise ValueError('hidden_size 和 num_heads 必须为正整数')
        if hidden_size % num_heads != 0:
            raise ValueError('hidden_size 必须能被 num_heads 整除')
        if num_kv_heads is None:
            num_kv_heads = num_heads
        if num_kv_heads <= 0 or num_heads % num_kv_heads != 0:
            raise ValueError('num_heads 必须能被 num_kv_heads 整除')

        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.num_key_value_groups = num_heads // num_kv_heads
        self.head_dim = hidden_size // num_heads
        self.scaling = self.head_dim ** -0.5

        self.q_proj = nn.Linear(hidden_size, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_size, num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_size, num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, hidden_size, bias=False)
        # 逐头归一化：作用在 head_dim 上，每个头独立校准。
        self.q_norm = HeadDimRMSNorm(self.head_dim)
        self.k_norm = HeadDimRMSNorm(self.head_dim)

    def _repeat_kv(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """把键值头扩展到与查询头相同的数量。

        Args:
            hidden_states: `[B, H_kv, S, D]` 的紧凑键值。
        Returns:
            `[B, H, S, D]` 的扩展键值。
        """
        if self.num_key_value_groups == 1:
            return hidden_states
        batch, num_kv_heads, slen, head_dim = hidden_states.shape
        hidden_states = hidden_states[:, :, None, :, :].expand(
            batch, num_kv_heads, self.num_key_value_groups, slen, head_dim
        )
        return hidden_states.reshape(batch, num_kv_heads * self.num_key_value_groups, slen, head_dim)

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        """返回与输入同形状的注意力输出。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
            attention_mask: 加性掩码，形状须可广播到
                `[batch, num_heads, seq_len, seq_len]`；可见位置为 0，屏蔽位置为 -inf。
        """
        batch, seq_len, _ = hidden_states.shape
        # 投影后先切成 [B, S, H, D] 再归一化，保证归一化只发生在 head_dim 上。
        query = self.q_proj(hidden_states).view(batch, seq_len, self.num_heads, self.head_dim)
        key = self.k_proj(hidden_states).view(batch, seq_len, self.num_kv_heads, self.head_dim)
        value = self.v_proj(hidden_states).view(batch, seq_len, self.num_kv_heads, self.head_dim)

        query = self.q_norm(query).transpose(1, 2)
        key = self.k_norm(key).transpose(1, 2)
        value = value.transpose(1, 2)

        # 键值扩展只用于当前计算，缓存里保存的仍是紧凑版本。
        key = self._repeat_kv(key)
        value = self._repeat_kv(value)

        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scaling
        if attention_mask is not None:
            scores = scores + attention_mask
        # softmax 在 FP32 中计算，避免低精度下概率分布失真。
        probs = F.softmax(scores, dim=-1, dtype=torch.float32).to(query.dtype)

        context = torch.matmul(probs, value)
        context = context.transpose(1, 2).contiguous().reshape(batch, seq_len, -1)
        return self.o_proj(context)


def sliding_window_mask(seq_len: int, window: int) -> torch.Tensor:
    """构造因果滑动窗口的加性掩码。

    窗口起点随查询位置前移：位置 t 只允许读取区间
    `[max(0, t - window + 1), t]` 内的词元，且始终包含当前位置。

    Args:
        seq_len: 序列长度。
        window: 窗口大小；小于 1 时退化为全因果可见。
    Returns:
        `[1, 1, seq_len, seq_len]` 的加性掩码，可见为 0、屏蔽为 -inf。
    """
    mask = torch.zeros(seq_len, seq_len, dtype=torch.bool)
    for query_pos in range(seq_len):
        start = max(0, query_pos - window + 1) if window > 0 else 0
        mask[query_pos, start:query_pos + 1] = True
    bias = torch.zeros(seq_len, seq_len, dtype=torch.float32)
    bias.masked_fill_(~mask, float('-inf'))
    return bias.reshape(1, 1, seq_len, seq_len)


class Qwen3DecoderLayer(nn.Module):
    """Qwen3 形态的解码器块：标准骨架加逐头归一化的注意力。

    块结构与 LLaMA3 同构：两次前置归一化、两次残差相加、注意力在前、
    MLP 在后。唯一的新增机制是注意力内部的逐头归一化，它不改变块外接口。
    """

    def __init__(self, hidden_size: int, intermediate_size: int, num_heads: int = 4,
                 num_kv_heads: int = 2):
        super().__init__()
        if hidden_size <= 0 or intermediate_size <= 0:
            raise ValueError('hidden_size 和 intermediate_size 必须为正整数')
        self.hidden_size = hidden_size
        self.input_layernorm = HeadDimRMSNorm(hidden_size)
        self.post_attention_layernorm = HeadDimRMSNorm(hidden_size)
        self.self_attn = QKNormAttention(hidden_size, num_heads, num_kv_heads)
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, hidden_states: torch.Tensor, attention_mask: torch.Tensor = None) -> torch.Tensor:
        """返回与输入同形状的解码器块输出。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
            attention_mask: 加性掩码，形状须可广播到
                `[batch, num_heads, seq_len, seq_len]`。
        """
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, attention_mask=attention_mask)
        hidden_states = residual + hidden_states

        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        gate = F.silu(self.gate_proj(hidden_states)) * self.up_proj(hidden_states)
        hidden_states = residual + self.down_proj(gate)
        return hidden_states


class AlternatingLayerStack(nn.Module):
    """按层类型交替使用滑动窗口与完整注意力的块堆叠。

    滑动窗口层负责局部依赖，完整注意力层负责全局依赖，两者交替出现。
    缓存规模的下降来自限制了一半的层所能访问的历史长度。
    """

    def __init__(self, hidden_size: int, intermediate_size: int, num_layers: int,
                 num_heads: int = 4, num_kv_heads: int = 2, window: int = 4):
        super().__init__()
        if num_layers <= 0:
            raise ValueError('num_layers 必须为正整数')
        self.window = window
        self.layers = nn.ModuleList(
            [Qwen3DecoderLayer(hidden_size, intermediate_size, num_heads, num_kv_heads)
             for _ in range(num_layers)]
        )

    def forward(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """按层顺序返回最终隐藏状态。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
        """
        seq_len = hidden_states.shape[1]
        for index, layer in enumerate(self.layers):
            if index % 2 == 0:
                # 偶数层使用滑动窗口；窗口外的历史按不可见处理。
                mask = sliding_window_mask(seq_len, self.window).to(hidden_states.device)
            else:
                # 奇数层保留完整的因果可见范围。
                mask = sliding_window_mask(seq_len, 0).to(hidden_states.device)
            hidden_states = layer(hidden_states, attention_mask=mask)
        return hidden_states


In [7]:
# Qwen3 机制测试：逐头归一化的尺度效果、归一化轴、滑动窗口边界与块契约。

def test_head_dim_norm_scales_down_large_values():
    """验证逐头归一化把大尺度输入压回单位量级，且不改变形状。"""
    torch.manual_seed(11)
    norm = HeadDimRMSNorm(head_dim=8)
    x = torch.randn(2, 3, 4, 8) * 50.0
    out = norm(x)
    assert out.shape == x.shape
    assert abs(out.float().std().item() - 1.0) < 0.2


def test_head_dim_norm_only_touches_last_dim():
    """验证归一化作用在 head_dim 上：按比例缩放结果一致，跨头互不影响。"""
    torch.manual_seed(12)
    norm = HeadDimRMSNorm(head_dim=4)
    # 形状 [batch=1, head=2, seq=2, head_dim=4]；两个头的输入刻意不同。
    x = torch.tensor([[
        [[1.0, 2.0, 3.0, 4.0], [4.0, 3.0, 2.0, 1.0]],
        [[0.5, 0.5, 0.5, 0.5], [2.0, 0.0, 2.0, 0.0]],
    ]])
    out = norm(x)
    # RMSNorm 对正比例缩放不变：同一头内，方向相同、幅度不同的输入得到相同结果。
    scaled = norm(x * 7.0)
    assert torch.allclose(out, scaled, atol=1e-5)
    # 归一化只沿最后一维进行，因此该维的均方根为 1（缩放参数初值为 1）。
    rms = out[0, 0, 0].float().pow(2).mean().sqrt()
    assert abs(rms.item() - 1.0) < 1e-4
    # 两个头的输入不同，归一化结果也不同，说明统计量没有跨头混合。
    assert not torch.allclose(out[0, 0, 0], out[0, 1, 0], atol=1e-5)


def test_qk_norm_restores_attention_score_scale():
    """验证逐头归一化把注意力分数的尺度拉回可训练区间。"""
    torch.manual_seed(13)
    attn = QKNormAttention(hidden_size=32, num_heads=4, num_kv_heads=2)
    x = torch.randn(1, 6, 32) * 20.0

    # 关掉归一化时的分数尺度。
    with torch.no_grad():
        attn.q_norm.weight.fill_(1.0)
        attn.k_norm.weight.fill_(1.0)
        query = attn.q_proj(x).view(1, 6, 4, 8).transpose(1, 2)
        key = attn.k_proj(x).view(1, 6, 2, 8).transpose(1, 2)
        key = attn._repeat_kv(key)
        raw_scores = torch.matmul(query, key.transpose(-2, -1)) * attn.scaling

        normalized_query = attn.q_norm(attn.q_proj(x).view(1, 6, 4, 8)).transpose(1, 2)
        normalized_key = attn._repeat_kv(
            attn.k_norm(attn.k_proj(x).view(1, 6, 2, 8)).transpose(1, 2)
        )
        norm_scores = torch.matmul(normalized_query, normalized_key.transpose(-2, -1)) * attn.scaling

    assert raw_scores.abs().max() > norm_scores.abs().max()
    assert norm_scores.abs().max() < 20.0
    # 归一化后的分数落在可训练区间，softmax 不至于退化为独热。
    probs = F.softmax(norm_scores, dim=-1)
    assert probs.max() < 1.0


def test_sliding_window_mask_boundary():
    """验证滑动窗口的起点随查询位置前移，且始终包含当前位置。"""
    mask = sliding_window_mask(6, window=3)
    assert mask.shape == (1, 1, 6, 6)
    # 位置 0 只能看自己。
    assert mask[0, 0, 0, 0] == 0.0 and torch.isinf(mask[0, 0, 0, 1])
    # 位置 2 可见 0、1、2，不可见 3。
    assert mask[0, 0, 2, 0] == 0.0 and mask[0, 0, 2, 2] == 0.0
    assert torch.isinf(mask[0, 0, 2, 3])
    # 位置 5 可见 3、4、5，不可见 0。
    assert torch.isinf(mask[0, 0, 5, 0])
    assert mask[0, 0, 5, 3] == 0.0 and mask[0, 0, 5, 5] == 0.0
    # 因果性：任何位置都看不到自己之后的位置。
    assert torch.isinf(mask[0, 0, 1, 2]) and torch.isinf(mask[0, 0, 4, 5])


def test_qwen3_decoder_layer_contract():
    """验证 Qwen3 块的形状、两次残差顺序与梯度连通性。"""
    torch.manual_seed(14)
    layer = Qwen3DecoderLayer(hidden_size=16, intermediate_size=24, num_heads=4, num_kv_heads=2)
    x = torch.randn(2, 5, 16, requires_grad=True)
    out = layer(x)
    assert out.shape == x.shape
    assert torch.isfinite(out).all()
    out.sum().backward()
    assert x.grad is not None
    assert all(param.grad is not None for param in layer.parameters())


def test_alternating_stack_uses_both_masks():
    """验证交替堆叠中滑动窗口层与完整注意力层都参与了计算。"""
    torch.manual_seed(15)
    stack = AlternatingLayerStack(hidden_size=16, intermediate_size=24, num_layers=4,
                                  num_heads=4, num_kv_heads=2, window=3)
    x = torch.randn(1, 6, 16)
    out = stack(x)
    assert out.shape == x.shape
    assert torch.isfinite(out).all()


def run_qwen3_tests():
    """运行 Qwen3 机制测试：逐头归一化、滑动窗口边界与块契约。"""
    test_head_dim_norm_scales_down_large_values()
    test_head_dim_norm_only_touches_last_dim()
    test_qk_norm_restores_attention_score_scale()
    test_sliding_window_mask_boundary()
    test_qwen3_decoder_layer_contract()
    test_alternating_stack_uses_both_masks()
    print("✅ Qwen3：逐头归一化尺度、滑动窗口边界与块契约测试通过")


run_qwen3_tests()


✅ Qwen3：逐头归一化尺度、滑动窗口边界与块契约测试通过


### 解析

#### 逐头归一化：作用维度决定语义

归一化本身是标准 RMSNorm，特殊之处只在作用维度。常规用法沿隐藏宽度归一化，而这里沿 `head_dim` 归一化，作用在查询与键已经拆成多头之后。这一步既保持了头的语义独立，也避免了把不同头的信息混进同一组统计量。

它要解决的故障出现在大模型训练中：查询与键的内积随规模增长，注意力分数变得很大，经过 softmax 后接近独热分布，而独热分布的 softmax 梯度接近于零，模型因此难以继续学习。把分数尺度拉回单位量级，softmax 重新具有区分度，梯度也重新出现。该归一化不改变任何张量形状，因此不触碰块外接口。

#### 滑动窗口交替：限制访问范围而非改变表示

滑动窗口掩码的窗口起点随查询位置前移，位置 `t` 只允许读取区间 `[max(0, t - window + 1), t]` 内的词元，且始终包含当前位置以维持因果性。

单独把所有层都改成滑动窗口会损失长程依赖能力，单独保留完整注意力又会让缓存规模随序列长度线性增长。交替堆叠的折中在于：滑动窗口层承担局部依赖，完整注意力层承担全局依赖，且前一层的输出已经通过残差把全局信息带入下一层，因此滑动窗口层并非真的与长程信息隔绝。

#### 与 DeepSeek-V4 的对照

两种改动都遵守同一条边界：块对外仍接受并返回 `[B, S, C]` 的主状态。差别在改动深度。Qwen3 只替换注意力内部的实现，骨架与残差形式完全不变，验证成本集中在分数尺度与窗口边界两处。V4 同时重写注意力、MLP 与残差接口，需要额外承担多流混合的双随机性、压缩条目的因果边界与位置修正等验证。

这一对照说明架构演进不等于推翻重来：多数前沿改动是保持接口不变的局部替换，而改动越靠近骨架，需要论证的不变量就越多。


In [8]:
class AttentionSinkAttention(nn.Module):
    """带可学习注意力汇的多头注意力，演示 softmax 归一化范围的扩展。

    汇（sink）是每个注意力头一个可学习标量，与真实键一起参与同一份 softmax，
    归一化后再把汇那一列丢弃。汇为模型提供「本头不关注任何真实词元」这一
    合法选项；它不是额外的键，也不携带任何内容。
    """

    def __init__(self, hidden_size: int, num_heads: int):
        super().__init__()
        if hidden_size <= 0 or num_heads <= 0:
            raise ValueError('hidden_size 和 num_heads 必须为正整数')
        if hidden_size % num_heads != 0:
            raise ValueError('hidden_size 必须能被 num_heads 整除')
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.scaling = self.head_dim ** -0.5
        self.q_proj = nn.Linear(hidden_size, num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(hidden_size, num_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(hidden_size, num_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(num_heads * self.head_dim, hidden_size, bias=False)
        # 每个头一个汇；初值为 0 表示训练初期等价于普通注意力。
        self.sinks = nn.Parameter(torch.zeros(num_heads))

    def forward(self, hidden_states: torch.Tensor, key_value_states: torch.Tensor = None,
                attention_mask: torch.Tensor = None) -> torch.Tensor:
        """返回与查询序列等长的注意力输出。

        可见集合可以比查询序列长：历史摘要条目会拼在查询之前，此时键序列长度
        大于查询序列长度。掩码的形状为 `[batch, num_heads, query_len, key_len]`。

        Args:
            hidden_states: `[batch, query_len, hidden_size]` 的查询来源。
            key_value_states: `[batch, key_len, hidden_size]` 的键值来源；
                缺省时与 `hidden_states` 相同。
            attention_mask: 加性掩码，形状须可广播到
                `[batch, num_heads, query_len, key_len]`；可见位置为 0，屏蔽位置为 -inf。
        """
        batch, query_len, _ = hidden_states.shape
        if key_value_states is None:
            key_value_states = hidden_states
        key_len = key_value_states.shape[1]
        q = self.q_proj(hidden_states).reshape(batch, query_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(key_value_states).reshape(batch, key_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(key_value_states).reshape(batch, key_len, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scaling
        if attention_mask is not None:
            scores = scores + attention_mask

        # 汇作为额外一列参与同一份 softmax，共享同一份概率质量。
        sinks = self.sinks.reshape(1, -1, 1, 1).expand(batch, -1, query_len, 1)
        combined = torch.cat([scores, sinks], dim=-1)
        # 先减去行最大值再归一化，避免低精度训练时 logits 溢出。
        combined = combined - combined.max(dim=-1, keepdim=True).values
        probs = F.softmax(combined, dim=-1)
        # 汇那一列只参与归一化，不参与内容聚合。
        probs = probs[..., :-1]

        context = torch.matmul(probs, v)
        context = context.transpose(1, 2).contiguous().reshape(batch, query_len, -1)
        return self.o_proj(context)


class SinkhornMixer(nn.Module):
    """把混合矩阵投影到双随机集合，构造流形约束的残差混合。

    双随机矩阵的每一行与每一列之和均为 1，因此它是无穷范数的压缩映射：
    ||D x||_inf <= ||x||_inf，等价地 D 的谱范数不超过 1。这是深层残差主干
    保持稳定的关键约束。注意该约束只覆盖无穷范数，二范数不受此限制。
    """

    def __init__(self, num_streams: int, eps: float = 1e-6, sinkhorn_iters: int = 8):
        super().__init__()
        if num_streams <= 0:
            raise ValueError('num_streams 必须为正整数')
        if sinkhorn_iters <= 0:
            raise ValueError('sinkhorn_iters 必须为正整数')
        self.num_streams = num_streams
        self.eps = eps
        self.sinkhorn_iters = sinkhorn_iters

    def project(self, weights: torch.Tensor) -> torch.Tensor:
        """把 `[..., N, N]` 的权重投影为双随机矩阵。

        Sinkhorn 交替归一化行和与列和，收敛速度随矩阵规模下降，因此迭代结束后
        再做一次行归一化，使行和精确等于 1；此时列和因总质量守恒而同样等于 1。

        Args:
            weights: 初始混合权重，形状 `[..., N, N]`。
        Returns:
            行和与列和均为 1 的混合矩阵，形状与输入一致。
        """
        mixed = F.softmax(weights, dim=-1)
        for _ in range(self.sinkhorn_iters):
            mixed = mixed / (mixed.sum(dim=-1, keepdim=True) + self.eps)
            mixed = mixed / (mixed.sum(dim=-2, keepdim=True) + self.eps)
        # 末次归一化取行方向：行和精确为 1，总质量守恒保证列和同样为 1。
        mixed = mixed / (mixed.sum(dim=-1, keepdim=True) + self.eps)
        return mixed


class MultiStreamResidual(nn.Module):
    """多流超连接：用三组学习权重替代单流的两次残差相加。

    主干从 `[B, S, D]` 扩展为 `[B, S, N, D]` 的 N 条并行流：

    - pre 把 N 条流压成 1 条，作为进入子层的输入；
    - post 把子层输出展开回 N 条流；
    - comb 以双随机矩阵混合输入流与输出流。

    残差项因此不再是简单相加，而是经过一个范数非扩张的线性变换。
    """

    def __init__(self, hidden_size: int, num_streams: int, eps: float = 1e-6, sinkhorn_iters: int = 8):
        super().__init__()
        if hidden_size <= 0 or num_streams <= 0:
            raise ValueError('hidden_size 和 num_streams 必须为正整数')
        self.hidden_size = hidden_size
        self.num_streams = num_streams
        self.mixer = SinkhornMixer(num_streams, eps=eps, sinkhorn_iters=sinkhorn_iters)
        # 三组权重由同一次投影产生；comb 需要 num_streams^2 个自由度。
        self.weight_proj = nn.Linear(hidden_size, 2 * num_streams + num_streams * num_streams, bias=False)

    def forward(self, hidden_streams: torch.Tensor):
        """返回展开权重、混合矩阵与压成单流的子层输入。

        三组权重由同一个投影层一次性算出。投影的输入是流本身而不是压平后的
        全部流：压平会把 N 条流的信息一次性混入，权重无法区分各流的贡献。
        这里先按流取均值得到一个稳定的读出量，再由投影层生成三组权重。

        Args:
            hidden_streams: `[batch, seq_len, num_streams, hidden_size]` 的并行流。
        Returns:
            `(post, comb, collapsed)`：
            post 形状 `[B, S, N]`，comb 形状 `[B, S, N, N]`，
            collapsed 形状 `[B, S, D]`。
        """
        batch, seq_len, num_streams, hidden_size = hidden_streams.shape
        if num_streams != self.num_streams or hidden_size != self.hidden_size:
            raise ValueError('hidden_streams 的流数与隐藏宽度必须与构造参数一致')

        readout = hidden_streams.mean(dim=2)
        raw = self.weight_proj(readout)
        pre_w, post_w, comb_w = raw.split([num_streams, num_streams, num_streams * num_streams], dim=-1)
        # pre 为正，post 允许放大到 2 倍，comb 在 Sinkhorn 投影后归一。
        pre = torch.sigmoid(pre_w) + 1e-6
        post = 2.0 * torch.sigmoid(post_w)
        comb = self.mixer.project(comb_w.reshape(batch, seq_len, num_streams, num_streams))

        collapsed = (pre.unsqueeze(-1) * hidden_streams).sum(dim=2)
        return post, comb, collapsed

    def merge(self, post: torch.Tensor, comb: torch.Tensor, sublayer_output: torch.Tensor,
              hidden_streams: torch.Tensor) -> torch.Tensor:
        """把子层输出展开回 N 条流并与输入流混合。

        Args:
            post: 展开权重，形状 `[B, S, N]`。
            comb: 双随机混合矩阵，形状 `[B, S, N, N]`。
            sublayer_output: 子层输出，形状 `[B, S, D]`。
            hidden_streams: 进入子层前的并行流，形状 `[B, S, N, D]`。
        Returns:
            形状 `[B, S, N, D]` 的新并行流。
        """
        expanded = post.unsqueeze(-1) * sublayer_output.unsqueeze(-2)
        # comb 按 `[j, k]` 索引输入流的首个流轴，故需转置后再相乘。
        residual = torch.matmul(comb.transpose(-1, -2), hidden_streams)
        return expanded + residual


class CompressedWindowAttention(nn.Module):
    """把连续词元按窗口压缩成少量条目，并按因果边界限制可见范围。

    每个窗口的 `compress_rate` 个词元被压缩成一条摘要条目；压缩比越大，
    历史键值占用的存储越少。压缩条目不落在当前词元位置，而是落在它所概括
    的那段历史的起始位置，因此因果边界必须按窗口而不是按词元计算。

    本类同时承担两件事：把当前窗口的历史压成摘要条目，以及把这些条目按
    因果边界拼回可见集合。它不做键值投影与注意力计算本身。
    """

    def __init__(self, hidden_size: int, compress_rate: int = 4):
        super().__init__()
        if hidden_size <= 0 or compress_rate <= 0:
            raise ValueError('hidden_size 和 compress_rate 必须为正整数')
        self.hidden_size = hidden_size
        self.compress_rate = compress_rate
        self.kv_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.gate_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        # 位置偏置是每个窗口槽位一份可学习标量，参与压缩时的门控打分。
        self.position_bias = nn.Parameter(torch.zeros(compress_rate, hidden_size))

    def compress(self, hidden_states: torch.Tensor):
        """把词元序列按窗口压缩成摘要条目。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
        Returns:
            `(compressed, window_positions)`：
            compressed 形状 `[B, n_windows, hidden_size]`，
            window_positions 形状 `[B, n_windows]`，记录每个窗口覆盖的起始位置。
        """
        batch, seq_len, _ = hidden_states.shape
        rate = self.compress_rate
        # 只压缩完整的窗口；不足一窗的尾部留给下一步。
        usable = (seq_len // rate) * rate
        if usable == 0:
            empty = hidden_states.new_zeros((batch, 0, self.hidden_size))
            return empty, hidden_states.new_zeros((batch, 0), dtype=torch.long)

        kv = self.kv_proj(hidden_states[:, :usable])
        gate = self.gate_proj(hidden_states[:, :usable])
        n_windows = usable // rate
        kv = kv.reshape(batch, n_windows, rate, self.hidden_size)
        gate = gate.reshape(batch, n_windows, rate, self.hidden_size) + self.position_bias

        # 门控 softmax 的权重把窗口内的词元合成一条摘要；低精度下用 FP32 计算以免近似并列。
        weights = gate.softmax(dim=2, dtype=torch.float32).to(kv.dtype)
        compressed = (kv * weights).sum(dim=2)

        positions = torch.arange(n_windows, device=hidden_states.device) * rate
        positions = positions.unsqueeze(0).expand(batch, -1)
        return compressed, positions

    def causal_bias(self, position_ids: torch.Tensor, window_positions: torch.Tensor) -> torch.Tensor:
        """按窗口边界给出可见性偏置。

        Args:
            position_ids: `[batch, seq_len]` 的当前词元绝对位置。
            window_positions: `[batch, n_windows]` 的压缩条目覆盖起点。
        Returns:
            `[batch, 1, seq_len, n_windows]` 的加性偏置，可见为 0、屏蔽为 -inf。
        """
        batch, seq_len = position_ids.shape
        n_windows = window_positions.shape[1]
        if window_positions.shape[0] != batch:
            raise ValueError('position_ids 与 window_positions 的批量大小必须一致')
        if n_windows == 0:
            return position_ids.new_zeros((batch, 1, seq_len, 0))

        # 查询位置 t 可以看到起点严格小于 t 的窗口；起点等于 t 的窗口尚未就绪。
        # window_positions [B, N] → [B, 1, 1, N]；position_ids [B, S] → [B, 1, S, 1]。
        visible = window_positions.reshape(batch, 1, 1, n_windows) < position_ids.reshape(batch, 1, seq_len, 1)
        bias = torch.zeros((batch, 1, seq_len, n_windows), dtype=torch.float32, device=position_ids.device)
        bias.masked_fill_(~visible, float('-inf'))
        return bias


class DeepseekV4DecoderLayer(nn.Module):
    """DeepSeek-V4 形态的解码器块：多流残差 + 压缩注意力 + 稀疏前馈。

    与 LLaMA3 形态的差别集中在三处：主干从单流变为 N 条并行流；注意力读取
    压缩后的历史条目；前馈按路由选择专家。块对外接口不变，仍接受并返回
    `[B, S, C]` 的主状态。
    """

    def __init__(self, hidden_size: int, num_streams: int = 4, compress_rate: int = 4,
                 num_experts: int = 4, top_k: int = 2):
        super().__init__()
        if hidden_size <= 0 or num_streams <= 0:
            raise ValueError('hidden_size 和 num_streams 必须为正整数')
        if num_experts <= 0 or top_k <= 0 or top_k > num_experts:
            raise ValueError('num_experts 必须为正数且 top_k 不得超过 num_experts')
        self.hidden_size = hidden_size
        self.num_streams = num_streams
        self.compress_rate = compress_rate
        self.num_experts = num_experts
        self.top_k = top_k

        self.attn_hc = MultiStreamResidual(hidden_size, num_streams)
        self.ffn_hc = MultiStreamResidual(hidden_size, num_streams)
        self.input_layernorm = nn.LayerNorm(hidden_size)
        self.post_attention_layernorm = nn.LayerNorm(hidden_size)
        self.attention = AttentionSinkAttention(hidden_size, num_heads=1)
        self.compressor = CompressedWindowAttention(hidden_size, compress_rate=compress_rate)
        # 专家为参数分组而不是独立模块，便于 CPU 上按索引批量执行。
        self.experts = nn.Parameter(torch.randn(num_experts, hidden_size, hidden_size) * 0.02)
        self.router = nn.Linear(hidden_size, num_experts, bias=False)

    def _route(self, hidden_states: torch.Tensor):
        """选择每个词元要激活的专家。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]`。
        Returns:
            `(indices, weights)`：indices 形状 `[B, S, top_k]`，weights 形状 `[B, S, top_k]`。
        """
        logits = self.router(hidden_states)
        scores = F.softmax(logits, dim=-1)
        weights, indices = scores.topk(self.top_k, dim=-1)
        weights = weights / (weights.sum(dim=-1, keepdim=True) + 1e-20)
        return indices, weights

    def _sparse_experts(self, hidden_states: torch.Tensor) -> torch.Tensor:
        """按路由结果执行专家前馈，并保持词元原有顺序。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]`。
        Returns:
            与输入同形状的前馈输出。
        """
        batch, seq_len, hidden_size = hidden_states.shape
        indices, weights = self._route(hidden_states)
        # index_add_ 只按第 0 维索引，因此把批量与序列维合并成一条扁平轴，
        # 累加后再还原成原形状。
        flat_states = hidden_states.reshape(batch * seq_len, hidden_size)
        output = flat_states.new_zeros((batch * seq_len, hidden_size))
        for expert_id in range(self.num_experts):
            # 找出路由到该专家的 (批量, 序列, top-k 槽位) 三元组。
            slot_mask = indices == expert_id
            if not slot_mask.any():
                continue
            hits = slot_mask.nonzero(as_tuple=False)
            token_idx = hits[:, 0] * seq_len + hits[:, 1]
            expert_weight = self.experts[expert_id]
            contribution = torch.tanh(flat_states[token_idx] @ expert_weight)
            contribution = contribution * weights[hits[:, 0], hits[:, 1], hits[:, 2], None]
            output.index_add_(0, token_idx, contribution.to(output.dtype))
        return output.reshape(batch, seq_len, hidden_size)

    def forward(self, hidden_states: torch.Tensor, position_ids: torch.Tensor = None) -> torch.Tensor:
        """返回与输入同形状的解码器块输出。

        注意力分支先按窗口压缩历史，把压缩条目与当前词元一起送入注意力，
        并用按窗口计算的偏置限制可见范围。

        Args:
            hidden_states: `[batch, seq_len, hidden_size]` 的主状态。
            position_ids: `[batch, seq_len]` 的绝对位置；缺省时按 0 递增构造。
        """
        batch, seq_len, hidden_size = hidden_states.shape
        if position_ids is None:
            position_ids = torch.arange(seq_len, device=hidden_states.device).unsqueeze(0).expand(batch, -1)

        # 主干扩展为 N 条并行流，各流初值相同。
        streams = hidden_states.unsqueeze(2).expand(-1, -1, self.num_streams, -1).contiguous()

        post, comb, collapsed = self.attn_hc(streams)
        # 压缩历史：把窗口内的词元合成摘要条目，并给出按窗口的可见性偏置。
        compressed, window_positions = self.compressor.compress(collapsed)
        window_bias = self.compressor.causal_bias(position_ids, window_positions)
        # 摘要条目与当前词元一起构成注意力的可见集合：
        # 查询只来自当前词元，键值来自摘要条目加当前词元。
        key_value_states = torch.cat([compressed, collapsed], dim=1)
        # 掩码的查询维是 seq_len，键维是 n_windows + seq_len；
        # 摘要条目按窗口偏置屏蔽，当前词元之间按因果关系屏蔽。
        token_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=collapsed.device), diagonal=1
        )
        token_bias = torch.zeros(seq_len, seq_len, dtype=torch.float32, device=collapsed.device)
        token_bias.masked_fill_(token_mask, float('-inf'))
        attention_mask = torch.cat(
            [window_bias, token_bias.reshape(1, 1, seq_len, seq_len).expand(batch, 1, seq_len, seq_len)], dim=-1
        )
        attn_output = self.attention(
            self.input_layernorm(collapsed),
            key_value_states=self.input_layernorm(key_value_states),
            attention_mask=attention_mask,
        )
        streams = self.attn_hc.merge(post, comb, attn_output, streams)

        post, comb, collapsed = self.ffn_hc(streams)
        mlp_output = self._sparse_experts(self.post_attention_layernorm(collapsed))
        streams = self.ffn_hc.merge(post, comb, mlp_output, streams)

        # 多流在块末按 pre 权重的语义压回单流，块外接口保持 `[B, S, C]`。
        return streams.mean(dim=2)


In [9]:
# V4 机制测试：注意力汇、双随机残差混合、压缩窗口因果边界。

def test_attention_sink_shares_probability_mass():
    """验证汇与真实键共享同一份概率质量，且不改变输出形状。"""
    torch.manual_seed(3)
    attn = AttentionSinkAttention(hidden_size=8, num_heads=2)
    x = torch.randn(2, 4, 8)
    out = attn(x)
    assert out.shape == x.shape

    # 汇的分数越高，真实键分到的概率质量越少。
    scores = torch.tensor([[[[2.0, 1.0]]]])
    sink_logits = torch.tensor([[-1.0, 0.0, 5.0]])
    for sink_value, expected in zip(sink_logits.flatten().tolist(), [0.0351, 0.09, 0.9362]):
        combined = torch.cat([scores, torch.full((1, 1, 1, 1), sink_value)], dim=-1)
        probs = F.softmax(combined, dim=-1)
        assert abs(probs.flatten()[-1].item() - expected) < 1e-3

    # 汇为 0 时，真实键拿到的概率质量与无汇情形一致。
    combined = torch.cat([scores, torch.zeros(1, 1, 1, 1)], dim=-1)
    probs_with_sink = F.softmax(combined, dim=-1)[..., :-1]
    probs_without = F.softmax(scores, dim=-1)
    assert torch.allclose(probs_with_sink, probs_without * probs_with_sink.sum(), atol=1e-6)


def test_sinkhorn_mixer_is_doubly_stochastic():
    """验证混合矩阵行和、列和均为 1，且不放大信号范数。"""
    torch.manual_seed(4)
    mixer = SinkhornMixer(num_streams=4)
    weights = torch.randn(2, 3, 4, 4)
    mixed = mixer.project(weights)
    # 行和由末次行归一化精确保证；列和由总质量守恒得到，允许 Sinkhorn 的残差。
    assert torch.allclose(mixed.sum(dim=-1), torch.ones(2, 3, 4), atol=1e-5)
    assert torch.allclose(mixed.sum(dim=-2), torch.ones(2, 3, 4), atol=1e-3)
    assert (mixed >= 0).all()

    # 双随机矩阵是无穷范数的压缩映射：||D x||_inf <= ||x||_inf。
    # l2 范数不受此约束，因此断言必须落在无穷范数上。
    x = torch.randn(4)
    assert (mixed @ x).abs().max() <= x.abs().max() + 1e-5

    # 更强的不变量：D^T D 的最大特征值不超过 1，即 D 的谱范数不超过 1。
    single = mixed[0, 0]
    spectral = torch.linalg.eigvals(single.T @ single).real.max()
    assert spectral <= 1.0 + 1e-4


def test_multi_stream_residual_shapes_and_norm():
    """验证多流残差的压流、展开与混合形状，并确认残差项非扩张。"""
    torch.manual_seed(5)
    residual = MultiStreamResidual(hidden_size=8, num_streams=4)
    streams = torch.randn(2, 3, 4, 8)
    post, comb, collapsed = residual(streams)
    assert post.shape == (2, 3, 4)
    assert comb.shape == (2, 3, 4, 4)
    assert collapsed.shape == (2, 3, 8)

    merged = residual.merge(post, comb, torch.randn(2, 3, 8), streams)
    assert merged.shape == streams.shape

    # 残差项由双随机矩阵混合而来，逐流的无穷范数不超过原流。
    residual_part = torch.matmul(comb.transpose(-1, -2), streams)
    assert (residual_part.abs().amax(dim=-1) <= streams.abs().amax(dim=-1) + 1e-5).all()


def test_compressed_window_causal_boundary():
    """验证压缩条目按窗口边界可见，而非按词元位置可见。"""
    torch.manual_seed(6)
    compressor = CompressedWindowAttention(hidden_size=8, compress_rate=4)
    x = torch.randn(2, 8, 8)
    compressed, positions = compressor.compress(x)
    assert compressed.shape == (2, 2, 8)
    # 第二个窗口覆盖词元 4-7，因此起点是 4 而不是 5 或 8。
    assert positions[0].tolist() == [0, 4]

    position_ids = torch.arange(8).unsqueeze(0).expand(2, -1)
    bias = compressor.causal_bias(position_ids, positions)
    assert bias.shape == (2, 1, 8, 2)
    # 位置 0 看不到任何窗口（起点 0 尚未就绪）；位置 4 能看到起点为 0 的窗口。
    assert torch.isinf(bias[0, 0, 0, 0])
    assert bias[0, 0, 4, 0] == 0.0
    assert torch.isinf(bias[0, 0, 4, 1])


def test_deepseek_v4_decoder_layer_contract():
    """验证 V4 解码器块的输入输出契约与梯度连通性。"""
    torch.manual_seed(7)
    layer = DeepseekV4DecoderLayer(hidden_size=16, num_streams=4, compress_rate=4,
                                   num_experts=4, top_k=2)
    x = torch.randn(2, 8, 16, requires_grad=True)
    out = layer(x)
    assert out.shape == x.shape
    assert torch.isfinite(out).all()
    out.sum().backward()
    assert x.grad is not None
    assert all(param.grad is not None for param in layer.parameters())


def run_deepseek_v4_tests():
    """运行 V4 机制测试：注意力汇、双随机混合与压缩因果边界。"""
    test_attention_sink_shares_probability_mass()
    test_sinkhorn_mixer_is_doubly_stochastic()
    test_multi_stream_residual_shapes_and_norm()
    test_compressed_window_causal_boundary()
    test_deepseek_v4_decoder_layer_contract()
    print("✅ DeepSeek-V4：注意力汇、双随机残差混合与压缩窗口因果边界测试通过")


run_deepseek_v4_tests()


✅ DeepSeek-V4：注意力汇、双随机残差混合与压缩窗口因果边界测试通过


### 解析

#### 注意力汇：softmax 归一化范围的扩展

普通注意力把全部概率质量分配给真实键。训练中会发现有些头并不需要读取任何真实词元，若没有可写的出口，它只能把注意力强行压在某个词元上。汇是每个头一个可学习标量，与真实键并列参与同一份 softmax，归一化后再把汇那一列丢弃。汇分数越高，真实键分到的概率质量越少；汇分数为负时几乎不吸收，退化为普通注意力。

汇必须与真实键共享同一份归一化：先拼成 `[..., seq_len + 1]` 一起做 softmax，再切掉最后一列。若改成先归一化真实键、再乘一个独立系数，两者的概率语义就不再一致。

#### 双随机残差混合：残差主干形态的约束

多流超连接把主干扩展成 N 条并行流，用三组学习权重驱动：pre 把 N 条流压成 1 条进入子层，post 把子层输出展开回 N 条，comb 以双随机矩阵混合输入流与输出流。

comb 的双随机性来自 Sinkhorn-Knopp 投影：交替归一化行和与列和，直到两者都收敛到 1。双随机矩阵的每一行非负且和为 1、每一列和也为 1，因此它是无穷范数的压缩映射：把任意向量映射后，各分量的绝对值都不超过映射前的最大绝对值。等价地说，该矩阵的谱范数不超过 1。普通残差相加没有这个约束，深层堆叠时信号强度可能逐层放大；受约束的混合则不会。

需要留意约束的适用范围：双随机性保证的是无穷范数而非二范数。逐流断言必须落在无穷范数上，写成二范数会得到错误结论。

comb 由投影产生，不保证对称。代码中混合项写作 `comb.transpose(-1, -2)` 与并行流相乘，因为索引关系是 `sum_j comb[j, k] * stream[j]`，方向不能与 `comb @ stream` 混用。

#### 压缩窗口的因果边界

压缩把连续词元合成少量摘要条目，压缩比越大、历史键值占用越少。摘要条目不落在当前词元位置，而落在它所覆盖的那段历史的起始位置，因此可见性必须按窗口起点判断：查询位置 t 可以看到起点严格小于 t 的窗口，起点等于 t 的窗口尚未覆盖到当前词元，不可见。

若沿用按词元的因果掩码，压缩后的键值序列长度已经与词元序列不同，掩码的行列含义会整体错位。这类问题在单层测试里不一定暴露，但在多轮追加历史时会累积成可见性错误。


## 相关阅读

完成最小块组装后，可先对照真实模型实现，再沿着残差主路径与 MoE 扩展理解哪些组件会被替换、哪些接口保持不变。

- [Transformers 中的 LLaMA 模型实现](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py)
- [LLaMA 原论文：LLaMA](https://arxiv.org/abs/2302.13971)
- [Block 与残差路径：架构专题](../topic_discussion/llm_architecture_evolution/06_block_residual_path.md)
- [Transformers 中的 Qwen3 模型实现](https://github.com/huggingface/transformers/blob/main/src/transformers/models/qwen3/modeling_qwen3.py)
- [Transformers 中的 DeepSeek-V4 模型实现](https://github.com/huggingface/transformers/blob/main/src/transformers/models/deepseek_v4/modeling_deepseek_v4.py)
- [06. MoE 路由器](../02_PyTorch_Algorithms/06_MoE_Router.ipynb)
- [08. 架构技巧](../02_PyTorch_Algorithms/08_Architecture_Tricks.ipynb)